# Week 4 Practice — Three Shapes of Data Collection Code

| Shape | Source | In this notebook |
|---|---|---|
| **Read once** | File · DB | `yellow_tripdata_2023-01.parquet` → pandas / SQLite + `read_sql` |
| **Ask repeatedly** | API | MOLIT (Korea Ministry of Land, Infrastructure and Transport) trip-volume-by-mode OpenAPI — `totalCount` tells you where the end is |
| **Receive endlessly** | Stream | Upbit WebSocket real-time crypto trades — **position (offset)** and **window** |

> Setup: `pip install -r requirements.txt` → run this notebook from top to bottom, in order

The file comes from the **"TLC Trip Record Data"** open dataset published by the New York City Taxi & Limousine Commission (NYC TLC).
- Official page: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
- Direct file URL (the download command in README.md): https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet



In [15]:
# Cell 0 — Common setup
import os, json, time, sqlite3, threading, queue
from pathlib import Path
import pandas as pd
import requests

for d in ["data", "raw", "raw/api", "state"]:
    Path(d).mkdir(parents=True, exist_ok=True)

PARQUET = "yellow_tripdata_2023-01.parquet"
DB_PATH = "data/taxi.db"
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
print("Ready")

Ready


---
## PART 1-A · Read once — File

Once you have it, you are done. The code runs a single time.
The hard part of collecting files is not reading them — it is checking **"is any day missing?"**

Apache Parquet is an open-source, column-oriented file format widely used for large-scale data analytics and processing.

Common formats such as CSV and JSON store data row by row. Parquet's defining feature is that it groups and stores data column by column.

![Parquet](Parquet.png)

In [16]:
# Cell 1 — Read the whole file
trips = pd.read_parquet(PARQUET)
print(trips.shape)
print(round(trips.memory_usage(deep=True).sum() / 1024**2), "MB")
trips.head(3)

(3066766, 19)
448 MB


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.3,1.0,0.5,0.0,0.0,1.0,14.3,2.5,0.0
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.9,1.0,0.5,4.0,0.0,1.0,16.9,2.5,0.0
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.9,1.0,0.5,15.0,0.0,1.0,34.9,2.5,0.0


### Column reference — one row = one taxi trip (19 columns)

These are the values the meter (TPEP device) records at the end of every trip. The column names become the column names of the DB table (`trips`) as they are, so refer to this table when you write SQL later.

**When · Where**

| Column | Type | Meaning |
|---|---|---|
| `VendorID` | int | Meter vendor that sent the record (1 = Creative Mobile Technologies, 2 = VeriFone) |
| `tpep_pickup_datetime` | datetime | Pickup time (when the meter was engaged) — the basis for date filters, the index, and incremental collection in this notebook |
| `tpep_dropoff_datetime` | datetime | Drop-off time (when the meter was disengaged) |
| `PULocationID` | int | Pickup zone number (PU = Pick-Up). TLC divides New York into taxi zones 1–265 |
| `DOLocationID` | int | Drop-off zone number (DO = Drop-Off) |

**Trip details**

| Column | Type | Meaning |
|---|---|---|
| `passenger_count` | float | Number of passengers — entered by the driver, so there are zeros and missing values |
| `trip_distance` | float | Trip distance, in **miles** (1 mile ≈ 1.6 km) |
| `RatecodeID` | float | Rate code: 1 = Standard, 2 = JFK flat fare, 3 = Newark airport, 4 = Nassau/Westchester, 5 = Negotiated fare, 6 = Group ride, 99 = Unknown |
| `store_and_fwd_flag` | str | Y = connection was lost, so the record was stored on the device and sent later; N = sent in real time |
| `payment_type` | int | Payment method: 1 = Card, 2 = Cash, 3 = No charge, 4 = Dispute, 0 = Not recorded |

**Fare (USD)**

| Column | Type | Meaning |
|---|---|---|
| `fare_amount` | float | Base meter fare (computed from time and distance) |
| `extra` | float | Extras — overnight and rush-hour surcharges, etc. |
| `mta_tax` | float | MTA (Metropolitan Transportation Authority) tax, $0.50 |
| `tip_amount` | float | Tip — recorded automatically for **card payments only**. Cash tips stay 0 |
| `tolls_amount` | float | Tolls (bridges and tunnels) |
| `improvement_surcharge` | float | Taxi improvement surcharge ($1 per trip; previously $0.30) |
| `congestion_surcharge` | float | Manhattan congestion surcharge, $2.50 |
| `airport_fee` | float | Airport pickup fee, $1.25 (pickups at LaGuardia and JFK) |
| `total_amount` | float | Total charged to the passenger — the sum of the items above. Cash tips are not included |

> **What a data collector should notice** (verify it yourself in the cell below)
> - `passenger_count` and `RatecodeID` look like integers but are **float** — these columns have rows with missing values, so they are stored as floating point.
> - In 71,743 rows, `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge` and `airport_fee` are **all empty at once**, and `payment_type` is 0 in every one of those rows.
> - The fare columns contain **negative values** too (refund and dispute records). At the collection stage we do not remove them — we keep them as they are.
> - To turn zone numbers into names, join with TLC's `taxi_zone_lookup.csv` (Taxi Zone Lookup Table).

In [ ]:
# Cell 1-1 — Column names · types · missing-value counts in one table
info = pd.DataFrame({"dtype": trips.dtypes.astype(str),      # data type of each column
                     "nulls": trips.isna().sum(),            # number of missing values (NaN)
                     "example": trips.iloc[0]})              # value in the first row (example)
print(info)
# What is payment_type in the rows with missing values? → all 0
print(trips.loc[trips["passenger_count"].isna(), "payment_type"].value_counts())

In [6]:
# Cell 2 — The file name says '2023-01', but does it really contain only January? Any missing days?
day = trips["tpep_pickup_datetime"].dt.date
counts = day.value_counts().sort_index()

in_jan = counts[(counts.index >= pd.Timestamp("2023-01-01").date()) &
                (counts.index <= pd.Timestamp("2023-01-31").date())]
print("Days in January:", len(in_jan), "/ 31")
print("Dates outside January (rows that slipped in):")
print(counts[~counts.index.isin(in_jan.index)])

Days in January: 31 / 31
Dates outside January (rows that slipped in):
tpep_pickup_datetime
2008-12-31     2
2022-10-24     4
2022-10-25     7
2022-12-31    25
2023-02-01    10
Name: count, dtype: int64


---
## PART 1-B · Read once — DB (try sending SQL)

With a file, there is nobody to ask "pick these out for me". With a DB, there is something on the other side that takes a question (SQL) and sends back **only the answer**.
For this practice we load the parquet into a single SQLite file (`data/taxi.db`), and from then on we only pull data out with SQL.

Three SQL words for today: `SELECT` which columns · `WHERE` which rows · `GROUP BY` grouped by what

In [ ]:
# Cell 3 — Load parquet → SQLite (first time only, 30 s to 1 min)
con = sqlite3.connect(DB_PATH)

exists = con.execute(
    "SELECT count(*) FROM sqlite_master WHERE type='table' AND name='trips'").fetchone()[0]
# Load only when the trips table does not exist yet (if it exists the count is non-zero, so we skip → re-running the cell does not load duplicates)
if not exists:
    # Record the start time — to measure how many seconds the load takes
    t0 = time.time()
    # Save the DataFrame as the 'trips' table. index=False: do not write the 0,1,2… row numbers as a column
    # chunksize=200_000: INSERT the 3.07 million rows 200,000 at a time (all at once is heavy on memory)
    trips.to_sql("trips", con, index=False, chunksize=200_000)
    # Create an index on the pickup-time column — speeds up date filters such as WHERE tpep_pickup_datetime >= ...
    con.execute("CREATE INDEX idx_pickup ON trips(tpep_pickup_datetime)")
    # Commit the changes to the DB file (data/taxi.db)
    con.commit()
    print(f"Load complete: {time.time() - t0:.1f} s")
print(con.execute("SELECT count(*) FROM trips").fetchone()[0], "rows in the DB")

In [ ]:
# Cell 4 — One line of read_sql: send a query, get a DataFrame back (LIMIT 5 = SQL's head())
df = pd.read_sql("SELECT * FROM trips LIMIT 5", con)
df

In [ ]:
# Cell 5 — One line of WHERE / GROUP BY decides 'how much comes back'. Which one takes longer? Why?
q_all = """SELECT * FROM trips
           WHERE tpep_pickup_datetime >= '2023-01-31'
             AND tpep_pickup_datetime <  '2023-02-01'"""

q_one = """SELECT payment_type,
                  COUNT(*)                    AS trips,
                  ROUND(SUM(total_amount), 0) AS sales
           FROM trips
           WHERE tpep_pickup_datetime >= '2023-01-31'
             AND tpep_pickup_datetime <  '2023-02-01'
           GROUP BY payment_type
           ORDER BY trips DESC"""

a = pd.read_sql(q_all, con)
b = pd.read_sql(q_one, con)
print(a.shape, a.memory_usage(deep=True).sum() // 1024, "KB")
print(b.shape, b.memory_usage(deep=True).sum() // 1024, "KB")
a

(100372, 19) 18731 KB
(5, 3) 0 KB


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-31 00:00:02,2023-01-31 00:08:20,1.0,1.93,1.0,N,263,238,1,11.4,1.0,0.5,2.00,0.0,1.0,18.40,2.5,0.00
1,2,2023-01-31 00:00:06,2023-01-31 00:18:36,1.0,6.79,1.0,N,132,134,1,31.0,1.0,0.5,6.70,0.0,1.0,41.45,0.0,1.25
2,2,2023-01-31 00:00:09,2023-01-31 00:30:40,1.0,11.98,1.0,N,186,188,1,49.9,1.0,0.5,6.10,0.0,1.0,61.00,2.5,0.00
3,2,2023-01-31 00:00:15,2023-01-31 00:06:31,1.0,1.93,1.0,N,90,48,2,10.0,1.0,0.5,0.00,0.0,1.0,15.00,2.5,0.00
4,2,2023-01-31 00:00:16,2023-01-31 00:09:28,1.0,2.78,1.0,N,249,246,1,13.5,1.0,0.5,3.70,0.0,1.0,22.20,2.5,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100367,2,2023-01-31 23:59:53,2023-02-01 00:02:02,1.0,0.91,1.0,N,141,263,1,5.8,1.0,0.5,1.00,0.0,1.0,11.80,2.5,0.00
100368,2,2023-01-31 23:59:54,2023-02-01 00:10:14,1.0,2.61,1.0,N,263,100,1,14.2,1.0,0.5,2.00,0.0,1.0,21.20,2.5,0.00
100369,2,2023-01-31 23:59:57,2023-02-01 00:06:29,1.0,3.04,1.0,N,132,216,2,13.5,1.0,0.5,0.00,0.0,1.0,17.25,0.0,1.25
100370,2,2023-01-31 23:59:59,2023-02-01 00:09:31,1.0,2.69,1.0,N,90,163,1,13.5,1.0,0.5,3.70,0.0,1.0,22.20,2.5,0.00


### Try sending SQL yourself

Run the cells below, changing only `q`. See the output of Cell 4 for the column names.
`payment_type`: 1 = card, 2 = cash, 3 = no charge, 4 = dispute · `PULocationID`/`DOLocationID`: pickup/drop-off zone number

In [ ]:
# Cell 6 — Trips and average fare by hour of day
# Build avg_total and avg_miles from total_amount and trip_distance
# Use tpep_pickup_datetime between '2023-01-01' and '2023-02-01'
q = """
SELECT strftime('%H', tpep_pickup_datetime) AS hour,
       COUNT(*)                            AS trips,
       ROUND(AVG(total_amount), 2)         AS avg_total,
       ROUND(AVG(trip_distance), 2)        AS avg_miles
FROM trips
WHERE
_____________
ORDER BY ____
"""
pd.read_sql(q, con)

,hour,trips,avg_total,avg_miles
0,00,84957,28.44,4.02
1,01,59799,26.06,3.49
2,02,42040,24.66,3.20
3,03,27437,25.73,3.74
4,04,17835,30.94,4.77
5,05,18011,36.06,15.30
6,06,43860,30.23,5.42
7,07,86876,26.68,5.25
8,08,116865,25.04,5.52
9,09,131110,25.29,3.12


In [ ]:
# Cell 7 — TOP 10 zones with the most pickups + tip ratio for card payments (tip / fare)
# payment_type = 1: card payments only, because cash tips are not recorded
# tip_amount (tip), fare_amount (fare), PULocationID (zone ID)
q = """
SELECT PULocationID,
       COUNT(*) AS trips,
       ROUND(100.0 * ________________, 1) AS tip_pct
FROM trips
WHERE payment_type = 1 AND ____?____ > 0
GROUP BY PULocationID
ORDER BY trips DESC
LIMIT ___
"""
pd.read_sql(q, con)

,PULocationID,trips,tip_pct
0,237,119111,25.0
1,236,112944,24.3
2,132,109380,18.4
3,161,108366,24.3
4,186,85945,23.6
5,162,85292,24.1
6,142,80690,24.6
7,230,74452,23.7
8,138,74342,23.4
9,170,71085,23.6


**Exercises** — write `q` yourself
- How many 'odd rows' are there where `trip_distance` is 0 or less, or `total_amount` is negative?
- What is the average `total_amount` of trips that have an airport fee (`airport_fee > 0`)?

In [ ]:
# Cell 8 — Don't fetch everything again every day: incremental collection
#   Run this cell several times. Each run fetches only 'one day's worth since last time'.
STATE = Path("state/taxi_last_ts.txt")
last_ts = STATE.read_text().strip() if STATE.exists() else "2022-12-31 23:59:59"  # where we stopped last time
until   = (pd.Timestamp(last_ts).normalize() + pd.Timedelta(days=1, hours=23, minutes=59, seconds=59))

new = pd.read_sql(
    """SELECT * FROM trips
       WHERE tpep_pickup_datetime > ? AND tpep_pickup_datetime <= ?
       ORDER BY tpep_pickup_datetime""",
    con, params=(last_ts, str(until)))

if len(new):                                             # 0 rows → leave the position unchanged
    out = f"raw/trips_after_{last_ts[:10]}.csv"
    new.to_csv(out, index=False)                         # save exactly as received
    STATE.write_text(str(new["tpep_pickup_datetime"].max()))  # record the position 'after' saving is done
    print(len(new), "rows transferred →", out)
print("Next starting point:", STATE.read_text() if STATE.exists() else last_ts)

> To start over from the beginning, delete `state/taxi_last_ts.txt`.
> This **"remembered value"** comes back in PART 3 under the name **position (offset)**.

---
## PART 1-C · Ask repeatedly — API (MOLIT trip volume by transport mode)

A loop that has an end. **Ask the server (totalCount) "are there more pages?"**

| Operation | Date parameter |
|---|---|
| `getDailyTransportationModeTripVolume` (daily) | `opr_ymd=20250801` |
| `getMonthlyTransportationModeTripVolume` (monthly) | `opr_ym=202508` |
| `getAnnualTransportationModeTripVolume` (annual) | `opr_yr=2025` |
| `getTransportationModeTripVolumeforPeoplewithReducedMobility` (people with reduced mobility) | `opr_ymd=20250801` |

Common: `ctpv_cd` (province/metropolitan-city code, e.g. 29 = Gwangju), `sgg_cd` (city/district code, e.g. 29140 = Seo-gu, Gwangju), `numOfRows` max 1000

In [ ]:
# Cell 9 — Settings and the fetch function
BASE_URL = "https://apis.data.go.kr/1613000/TransportationModeTripVolume"

def load_key(name="DATA_GO_KR_KEY"):
    """Never write the key in code — read it from the environment variable, or else from the .env file (not committed to git)."""
    if os.getenv(name):
        return os.environ[name]
    if Path(".env").exists():
        for line in Path(".env").read_text(encoding="utf-8").splitlines():
            k, _, v = line.partition("=")
            if k.strip() == name and v.strip():
                return v.strip().strip("\"'")
    raise RuntimeError(f"{name} is not set — copy .env.example to .env and put your key in it")

SERVICE_KEY = load_key()

def fetch(operation, params):
    """Request one page and return it as a dict."""
    p = {"serviceKey": SERVICE_KEY, "dataType": "JSON", **params}
    r = requests.get(f"{BASE_URL}/{operation}", params=p, timeout=30)
    try:
        return r.json()           # errors often arrive in a JSON envelope too → check() decides
    except ValueError:
        r.raise_for_status()      # if it is not JSON, look at the HTTP error first
        raise

OP = "getDailyTransportationModeTripVolume"
QUERY = {"opr_ymd": "20250801", "ctpv_cd": "29", "sgg_cd": "29140"}   # 2025-08-01, Seo-gu, Gwangju

data = fetch(OP, {**QUERY, "pageNo": 1, "numOfRows": 3})
Path("raw/api/sample.json").write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(data, ensure_ascii=False, indent=2)[:900])   # look at the structure before opening the envelope

**Think about it** — what if `status_code` is 200 but the table is empty? → The HTTP status alone is not enough; you have to look at the **header and body inside the response**.

- Normal: `Response.header.resultCode == "200"`
- Page number past the end: 200 · SUCCESS, but `item` is an empty list
- Wrong key: a completely different envelope (`OpenAPI_ServiceResponse`) comes back

In [ ]:
# Cell 10 — Checking the response: read the status of the response first
def check(data):
    if "OpenAPI_ServiceResponse" in data:                  # gateway errors such as an invalid service key
        h = data["OpenAPI_ServiceResponse"]["cmmMsgHeader"]
        raise RuntimeError(f"[{h['returnReasonCode']}] {h['errMsg']} ({h['returnAuthMsg']})")
    header = data["Response"]["header"]
    if header["resultCode"] not in ("00", "200"):
        raise RuntimeError(f"[{header['resultCode']}] {header['resultMsg']}")
    return data["Response"]["body"]

# ① A page past the end — status 200, SUCCESS, and yet the table is empty
r = requests.get(f"{BASE_URL}/{OP}", timeout=30,
                 params={"serviceKey": SERVICE_KEY, "dataType": "JSON", **QUERY,
                         "pageNo": 99, "numOfRows": 1000})
empty = check(r.json())
print("① status_code:", r.status_code, "/ items:", empty["items"]["item"], "/ totalCount:", empty["totalCount"])

# ② Wrong key — the response status is different
r = requests.get(f"{BASE_URL}/{OP}", timeout=30,
                 params={"serviceKey": "WRONG_KEY", "dataType": "JSON", **QUERY})
print("② status_code:", r.status_code)
try:
    check(r.json())
except RuntimeError as e:
    print("   caught →", e)

body = check(data)
print("OK → totalCount:", body["totalCount"], "/ pageNo:", body["pageNo"], "/ numOfRows:", body["numOfRows"])

In [ ]:
# Cell 11 — Open three levels, then json_normalize in one line
items = data["Response"]["body"]["items"]["item"]      #  Response → body → items → item
df = pd.json_normalize(items)
print(df.head(3))
print(df.dtypes)          # code values ('01', '05') arrive as strings, head counts as numbers — keep them as strings to preserve the leading 0!

In [ ]:
# Cell 12 — Multiple pages: don't hard-code the end, ask the server (totalCount)
def fetch_all(operation, query, num_rows=1000, save_prefix=None):
    rows, page = [], 1
    while True:                                           # we don't know where the end is
        body = check(fetch(operation, {**query, "pageNo": page, "numOfRows": num_rows}))
        item = body["items"]["item"] if body["items"] else []
        if isinstance(item, dict):                        # some APIs return a dict, not a list, when there is a single record
            item = [item]
        if save_prefix:                                   # save to raw/ exactly as received
            Path(f"{save_prefix}_p{page}.json").write_text(
                json.dumps(body, ensure_ascii=False), encoding="utf-8")
        rows += item
        print(f"  page {page}: {len(item)} records (cumulative {len(rows)} / {body['totalCount']})")
        if not item or page * num_rows >= body["totalCount"]:   # the end
            break
        page += 1
        time.sleep(0.3)                                   # be polite to the server
    return pd.DataFrame(rows), body["totalCount"]

daily, total = fetch_all(OP, QUERY, save_prefix="raw/api/daily_20250801_29140")
print(len(daily), "records /", total, "records")

In [ ]:
# Cell 13 — A quick look at the table we got: trips generated, by transport mode × time of day
#   trfc_mns_nm = transport mode name, users_type_nm = user type name,
#   ocrn_pasg_nope = number of passengers (trips generated), tzon = time-of-day band
#   The values in these columns are Korean text, exactly as the API returns them.
print(daily.groupby("trfc_mns_nm")["ocrn_pasg_nope"].sum().sort_values(ascending=False))
print()
print(daily.groupby("users_type_nm")["ocrn_pasg_nope"].sum().sort_values(ascending=False))
daily.pivot_table(index="tzon", columns="trfc_mns_nm", values="ocrn_pasg_nope", aggfunc="sum").fillna(0).astype(int)

**Exercises**
1. Change `OP` to monthly (`getMonthlyTransportationModeTripVolume`) and `QUERY` to `{"opr_ym": "202508", ...}` — how many pages do you get?
2. If you reduce `numOfRows` to 100, how many pages? What about 1001? (Guide: max 1000)
3. Fetch other dates/regions and pile them up in `raw/api/`. Use the number of files you received and `totalCount` to check that no page is missing.

---
## PART 3 · Receive endlessly — Real-time crypto prices (Upbit WebSocket)

A loop with no end. **You must remember "how far have I received?"**

- Source: `wss://api.upbit.com/websocket/v1` — receives real-time trades without an API key
- **Producer** (WebSocket receiver) → **Broker** (`queue.Queue` / the append-only ledger `raw/upbit_trades.jsonl`) → **Consumer** (aggregation)
- The consumer writes its **position (offset)** under `state/`, and even if it dies and comes back it reads from that spot
- To aggregate an endless flow, you cut it into **windows**

> To really run it 'endlessly' in a terminal, use `upbit_producer.py` / `upbit_consumer.py` (see the README)

In [ ]:
# Cell 14 — Look at the structure of the first message
import uuid, certifi, websocket   # websocket-client

WS_URL = "wss://api.upbit.com/websocket/v1"
CODES = ["KRW-BTC", "KRW-ETH", "KRW-XRP", "KRW-SOL", "KRW-DOGE"]
SSLOPT = {"ca_certs": certifi.where()}   # avoids the certificate problem with the python.org installer on macOS

def subscribe(codes=CODES):
    ws = websocket.create_connection(WS_URL, timeout=10, sslopt=SSLOPT)
    ws.send(json.dumps([{"ticket": str(uuid.uuid4())},
                        {"type": "trade", "codes": codes, "is_only_realtime": True},
                        {"format": "DEFAULT"}]))
    return ws

ws = subscribe()
msg = json.loads(ws.recv())     # Upbit sends bytes — json.loads handles that for us
ws.close()
print(json.dumps(msg, ensure_ascii=False, indent=2))

Key fields: `code` market · `trade_price` trade price · `trade_volume` trade volume · `ask_bid` sell/buy · `trade_timestamp` trade time (ms) · `sequential_id` unique trade ID (for de-duplication)

In [ ]:
# Cell 15 — Producer → queue.Queue → Consumer: if processing is slow, the queue piles up
q = queue.Queue()                       # plays the role of the broker
stop = threading.Event()

def producer():                         # producer: just put() whatever arrives
    ws = subscribe()
    while not stop.is_set():
        try:
            q.put(json.loads(ws.recv()))
        except websocket.WebSocketTimeoutException:
            continue
    ws.close()

def consumer(per_sec=2):                # consumer: suppose it can only process 2 messages per second
    while not stop.is_set():
        try:
            q.get(timeout=1)
        except queue.Empty:
            continue
        time.sleep(1 / per_sec)         # time spent processing → try changing this number
        q.task_done()

threading.Thread(target=producer, daemon=True).start()
threading.Thread(target=consumer, daemon=True).start()
for t in range(15):
    time.sleep(1)
    print(f"{t+1:2d} s  messages waiting in the queue: {q.qsize()}")
stop.set()

The real trade rate is not constant — when the market is hot the queue grows fast, and when it is quiet the consumer catches up.
Try raising `per_sec` to 20, or attach one more consumer thread.

With `queue.Queue`, **once you take an item out, it is gone.** So a second consumer cannot read the same data again.
→ From the next cell on, we use an **append-only ledger** (a jsonl file) as the broker.

In [ ]:
# Cell 16 — Producer in the background: it only appends to the ledger (raw/upbit_trades.jsonl)
LEDGER = Path("raw/upbit_trades.jsonl")
ledger_stop = threading.Event()

def ledger_producer(seconds=120):
    ws, end = subscribe(), time.time() + seconds
    with LEDGER.open("a", encoding="utf-8") as f:
        while not ledger_stop.is_set() and time.time() < end:
            try:
                m = json.loads(ws.recv())
            except websocket.WebSocketTimeoutException:
                continue
            f.write(json.dumps(m, ensure_ascii=False) + "\n")   # one line = one message
            f.flush()
    ws.close()

LEDGER.touch()
ledger_stop.clear()
threading.Thread(target=ledger_producer, args=(120,), daemon=True).start()
time.sleep(5)
print("Appending to the ledger for 2 minutes — run the cells below several times, a few seconds apart")

In [ ]:
# Cell 17 — Each consumer group has its own position: seek(position) → read only the new lines → record tell() after processing
def consume(group, max_lines=None):
    pos_file = Path(f"state/upbit_pos_{group}.txt")
    pos = int(pos_file.read_text()) if pos_file.exists() else 0      # last position (bytes)
    rows = []
    with LEDGER.open("rb") as f:
        f.seek(pos)                                                  # jump to that spot
        while max_lines is None or len(rows) < max_lines:
            line = f.readline()
            if not line.endswith(b"\n"):                             # a half-written line → leave it for next time
                break
            rows.append(json.loads(line))
            pos = f.tell()
    # --- write the position 'after' the processing (aggregating, saving) is finished here ---
    pos_file.write_text(str(pos))
    return pd.DataFrame(rows), pos

new_a, pos_a = consume("A")
print(f"[A] {len(new_a)} new, position {pos_a} bytes / ledger size {LEDGER.stat().st_size}")
if len(new_a):
    print(new_a.groupby("code")["trade_volume"].agg(["count", "sum"]))

In [ ]:
# Cell 18 — Group B reads from its own position regardless of A, only 5 at a time
new_b, pos_b = consume("B", max_lines=5)
print(f"[B] {len(new_b)} new, position {pos_b}")
print(new_b[["code", "trade_price", "trade_volume", "ask_bid", "sequential_id"]] if len(new_b) else "none")
print({p.stem: int(p.read_text()) for p in Path("state").glob("upbit_pos_*.txt")})

- Run A several times and it reads only what is newly appended, while B follows separately at its own pace — **the ledger stays as it is; the position lives on the reader's side.**
- To read again from the start, delete `state/upbit_pos_A.txt` (a new group C = starts from 0).
- If you write the position **before** processing, you lose data when you die; if you write it **after**, data may be processed twice → de-duplicate with `sequential_id`.

In [ ]:
# Cell 19 — Window: cut the endless flow into 10-second pieces and aggregate VWAP and volume per market
#   Receives 'endlessly' for max_seconds, then stops. To stop midway, press ■ (interrupt)
WINDOW_SEC = 10

def window_stream(max_seconds=40, codes=CODES):
    ws, end = subscribe(codes), time.time() + max_seconds
    window, cur, seen = {}, None, set()
    try:
        while time.time() < end:                                   # effectively while True
            try:
                m = json.loads(ws.recv())
            except websocket.WebSocketTimeoutException:
                continue
            if m["sequential_id"] in seen:                         # ignore a trade that arrives twice
                continue
            seen.add(m["sequential_id"])
            w = m["trade_timestamp"] // 1000 // WINDOW_SEC * WINDOW_SEC   # start time of the window (seconds)
            if cur is not None and w > cur:                        # next window starts → close the earlier windows
                for done in sorted(k for k in window if k < w):
                    closed = pd.DataFrame(window.pop(done)).T.sort_index()
                    closed["vwap"] = (closed["amount"] / closed["volume"]).round(0)
                    closed["trades"] = closed["trades"].astype(int)
                    print(f"\n[{pd.to_datetime(done, unit='s', utc=True).tz_convert('Asia/Seoul'):%H:%M:%S}] window closed")
                    print(closed[["trades", "volume", "vwap", "last"]])
            cur = w if cur is None else max(cur, w)                # a late trade does not move the window back
            s = window.setdefault(w, {}).setdefault(
                m["code"], {"trades": 0, "volume": 0.0, "amount": 0.0, "last": 0.0})
            s["trades"] += 1
            s["volume"] += m["trade_volume"]
            s["amount"] += m["trade_price"] * m["trade_volume"]
            s["last"] = m["trade_price"]
    except KeyboardInterrupt:
        print("Interrupted")
    finally:
        ws.close()
    print(f"\n(the last window is still open — aggregating {len(window.get(cur, {}))} markets)")

window_stream(max_seconds=40)

**Exercises**
1. Set `WINDOW_SEC` to 60 — compare with the result of aggregating the ledger from Cell 17 as a batch (`groupby` + `dt.floor("1min")`)
2. Also output the share of `ask_bid == "BID"` (buys) within each window
3. When a window closes, append the result to `raw/upbit_windows.jsonl` — stream results end up as files (batch) too

In [ ]:
# Cell 20 — Cleanup
ledger_stop.set()
con.close()
print("Ledger:", LEDGER.stat().st_size if LEDGER.exists() else 0, "bytes")